# AHA-Fed — real-data experiment suite (Kaggle, T4)

Runs every experiment the paper needs on **real MedMNIST v2 data** — no synthetic benchmark.

**Before running:** Settings → Accelerator → **GPU T4 x2** (or T4), and Internet → **On**.

Only one T4 is used — the federated clients train sequentially, so the second GPU sits idle.
Pick `T4 x2` anyway if offered; it costs the same quota.

Results are written to `/kaggle/working/runs/paper/real/` and zipped at the end for download.

## 1. Get the code

In [ ]:
# The repo is attached as the private Kaggle Dataset 'aha-fed-code'.
# Kaggle mounts CLI-pushed private datasets under a nested path
# (/kaggle/input/datasets/<owner>/<slug>/), so search rather than assume.
import glob, os, shutil, sys, zipfile

REPO = '/kaggle/working/repo'

def find_source():
    if not os.path.isdir('/kaggle/input'):
        raise SystemExit('No /kaggle/input — attach the dataset first.')
    # Any directory containing paper/build_paper.py is the repo root.
    for hit in glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True):
        return os.path.dirname(os.path.dirname(hit)), 'tree'
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        return zips[0], 'zip'
    found = glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
    raise SystemExit(
        'Dataset attached but no repo found. Right panel -> Add Input -> '
        f'aha-fed-code, then re-run. Saw: {found[:10]}')

if not os.path.isdir(os.path.join(REPO, 'paper')):
    src, kind = find_source()
    os.makedirs(REPO, exist_ok=True)
    if kind == 'zip':
        with zipfile.ZipFile(src) as z:
            z.extractall(REPO)
    else:
        shutil.copytree(src, REPO, dirs_exist_ok=True)
    print(f'staged repo from {src} ({kind})')

os.chdir(REPO)
sys.path.insert(0, REPO)
assert os.path.isdir('paper'), sorted(os.listdir(REPO))[:20]
print('cwd:', os.getcwd())
print('ok :', sorted(os.listdir(REPO))[:12])

## 2. Dependencies

In [ ]:
!pip install -q medmnist opacus

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f'gpu {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
    print('using gpu 0 only (clients train sequentially)')
else:
    print('WARNING: no GPU — enable it in Settings > Accelerator')

## 3. Sanity check

Two rounds on PneumoniaMNIST to confirm data downloads and training runs before committing hours of compute.

In [ ]:
import time
from pathlib import Path
from paper.run_real_experiments import run_one, OUT_DIR

OUT_DIR.mkdir(parents=True, exist_ok=True)
smoke = OUT_DIR / '_smoketest.json'
smoke.unlink(missing_ok=True)

t = time.time()
run_one(smoke, aggregator='fedavg', dataset='pneumoniamnist',
        num_classes=2, samples_per_client=300, num_rounds=2, seed=42)
print(f'{(time.time()-t)/2:.1f}s per round')
smoke.unlink(missing_ok=True)

## 4. Real-data suite

Replaces every synthetic experiment. **63 runs total, roughly 2.5–3 h on a T4** —
inside Kaggle's 9 h session limit, but run the cells in order so the most important
matrix finishes first.

| Matrix | Runs | What it replaces | Dataset |
|---|---|---|---|
| `--noise` | 32 | synthetic noise sweep (**the headline claim**) | PathMNIST 3 seeds + PneumoniaMNIST |
| `--pneu`  | 15 | synthetic multi-seed | PneumoniaMNIST, 5 aggregators x 3 seeds |
| `--het`   | 10 | synthetic heterogeneity sweep | PathMNIST |
| `--scale` | 6  | synthetic client scaling | PathMNIST, K = 4/8/12 |

Runs are skipped if their JSON already exists, so a session that times out can be
resumed by re-running the same cell.

**Save a Version (Save & Run All) rather than an interactive session** if you want
it to survive a disconnect.

In [ ]:
!python paper/run_real_only_experiments.py --noise

In [ ]:
!python paper/run_real_only_experiments.py --pneu

In [ ]:
!python paper/run_real_only_experiments.py --het --scale

## 5. Re-run the existing PathMNIST / DP / hyperparameter matrices

Optional — these already have results in the repo. Run only if you want them regenerated on this GPU.

In [ ]:
# !python paper/run_real_experiments.py --real --dp --hp

## 6. Check what the new results support

In [ ]:
!python paper/verify_claims.py

In [ ]:
import glob, json, statistics as st

def final(f):
    d = json.load(open(f))
    return d.get('final') or (d.get('history') or [{}])[-1]

print('=== PathMNIST label-noise sweep (3 seeds) ===')
for rate in ['0.0', '0.3', '0.5', '0.7']:
    row = []
    for m in ['aha', 'fedavg']:
        e = [final(f)['ece'] for f in sorted(glob.glob(
            f'runs/paper/real/real_noise_{m}_n{rate}_s*.json'))]
        row.append(f"{m} {st.mean(e):.4f}+/-{st.pstdev(e):.4f}" if e else f'{m} -')
    print(f'  noise={rate}  ' + '   '.join(row))

print('\n=== PneumoniaMNIST (chest X-ray), 3 seeds ===')
for m in ['aha_fed', 'fedavg', 'fedprox', 'fedbn', 'feddisco']:
    fs = sorted(glob.glob(f'runs/paper/real/pneu_{m}_s*.json'))
    if not fs:
        continue
    v = [final(f) for f in fs]
    ece = [x['ece'] for x in v]
    auroc = [x['auroc'] for x in v if x.get('auroc') is not None]
    au = f"{st.mean(auroc):.4f}" if auroc else 'n/a'
    print(f'  {m:10s} ECE {st.mean(ece):.4f}+/-{st.pstdev(ece):.4f}  AUROC {au}')

## 7. Download results

Zip lands in `/kaggle/working` — grab it from the Output tab, unzip over `runs/paper/real/` locally, then rebuild the paper.

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/aha_fed_results.zip runs/paper/real
!ls -lh /kaggle/working/aha_fed_results.zip
!ls /kaggle/working/repo/runs/paper/real | wc -l